In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

# add gsl includes to root
environ["ROOT_INCLUDE_PATH"] = environ["ROOT_INCLUDE_PATH"] + ":" + environ["GSL_ROOT_DIR"] + "/include"

In [2]:
import ROOT
from analysis_framework import Dataset
from ObjectSelectionHelper import ObjectSelectionHelper

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x9721610
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x99b35a0


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 6
no_rvec = True
# write_outputs = False
write_outputs = True

output_collections = r"(postfit_\w+_lvec)|(iso_lep_charge)"
dataset_path = "data/datasets/selected-objects/signal-only.json"
output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/fitted-objects/signal-only"
output_meta_path = "data/datasets/fitted-objects"
output_meta = f"{output_meta_path}/signal-only-clean.json"
# plot_dir = "plots/pre-selection/full"


In [4]:
if not write_outputs:
    ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ObjectSelectionHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xd110850


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
ROOT.gInterpreter.Declare("#include \"kinfit.h\"")
ROOT.gSystem.Load("libMarlinKinfit.so")
# E_err = 4.4
# E_err = 3.5
# Theta_err = 0.045
# Phi_err = 0.04
# values for with BIB
# E_err = [0.85, 5.3, 3.5, 3.7]
# Theta_err = [3.3e-5, 0.71, 0.045, 0.035]
# Phi_err = [8.4e-5, 0.52, 0.039, 0.032]
# values for clean
# E_err = [0.85, 5.0, 3.2, 3.4]
# Theta_err = [3.3e-5, 0.70, 0.040, 0.030]
# Phi_err = [8.4e-5, 0.53, 0.039, 0.033]
# test values for clean, smaller electron E_err to disregard the long tail
# bigger values for nu as technically it is free and purely derived from the others
E_err = [0.85/4, 5.0 * 10, 3.2, 3.4]
Theta_err = [3.3e-5, 0.70 * 10, 0.040 * 0.9, 0.030 * 0.9]
Phi_err = [8.4e-5, 0.53 * 10, 0.039 * 0.9, 0.033 * 0.9]

E_cms = 250.
m_W = 80.419
width_W = 2.049
fitter = ROOT.enuWFit(E_err, Theta_err, Phi_err, x_angle, E_cms, m_W, width_W)

In [9]:
# reco_columns = ["iso_lep_lvec", "nu_lvec", "R2Jet_sel1_lvec", "R2Jet_sel2_lvec"]
reco_columns = ["iso_lep_lvec", "clean_nu_lvec", "clean_R2Jet_sel1_lvec", "clean_R2Jet_sel2_lvec"]

In [10]:
# analysis.add_filter("clean_R2Jet_sel2_lvec.P() > 0.", "jet2.P > 0.")
# analysis.add_filter("clean_R2Jet_sel1_lvec.P() > 0.", "jet1.P > 0.")
# analysis.add_filter("clean_nu_lvec.P() > 0.", "nu.P > 0.")

In [11]:
analysis.Define("can_do_fit", "clean_nu_lvec.P() > 0. && clean_R2Jet_sel1_lvec.P() > 0. && clean_R2Jet_sel2_lvec.P() > 0.")
analysis.Define("fitres", fitter, reco_columns + ["can_do_fit"])
analysis.Define("prob", "fitres.prob")
analysis.Define("chi2", "fitres.chi2")
analysis.Define("error", "fitres.error")
analysis.Define("postfit_iso_lep_lvec", "fitres.obj1")
analysis.Define("postfit_nu_lvec", "fitres.obj2")
analysis.Define("postfit_R2Jet1_lvec", "fitres.obj3")
analysis.Define("postfit_R2Jet2_lvec", "fitres.obj4")

In [12]:
analysis.Define("prefit_leptonic_W_M", f"({reco_columns[0]} + {reco_columns[1]}).M()")
analysis.Define("prefit_hadronic_W_M", f"({reco_columns[2]} + {reco_columns[3]}).M()")

analysis.Define("postfit_leptonic_W_M", "(postfit_iso_lep_lvec + postfit_nu_lvec).M()")
analysis.Define("postfit_hadronic_W_M", "(postfit_R2Jet1_lvec + postfit_R2Jet2_lvec).M()")

In [13]:
analysis.define_deltas("postfit_iso_lep", "postfit_iso_lep_lvec", "true_lep_lvec", categories=signal_category)
analysis.define_deltas("postfit_nu", "postfit_nu_lvec", "true_nu_lvec", categories=signal_category)
analysis.define_deltas("postfit_R2Jet1", "postfit_R2Jet1_lvec", "true_quark1_lvec", categories=signal_category)
analysis.define_deltas("postfit_R2Jet2", "postfit_R2Jet2_lvec", "true_quark2_lvec", categories=signal_category)

analysis.define_deltas("iso_lep", "iso_lep_lvec", "true_lep_lvec", categories=signal_category)
analysis.define_deltas("nu", "nu_lvec", "true_nu_lvec", categories=signal_category)
analysis.define_deltas("R2Jet1", "R2Jet1_lvec", "true_quark1_lvec", categories=signal_category)
analysis.define_deltas("R2Jet2", "R2Jet2_lvec", "true_quark2_lvec", categories=signal_category)
# analysis.define_deltas("nu", "clean_nu_lvec", "true_nu_lvec", categories=signal_category)
# analysis.define_deltas("R2Jet1", "clean_R2Jet1_lvec", "true_quark1_lvec", categories=signal_category)
# analysis.define_deltas("R2Jet2", "clean_R2Jet2_lvec", "true_quark2_lvec", categories=signal_category)

In [14]:
analysis.book_histogram_1D("prob", "prob", ("", "", 100, 0., 1.))
analysis.book_histogram_1D("chi2", "chi2", ("", "", 100, 0., 50.))
analysis.book_histogram_1D("error", "error", ("", "", 20, -10., 10.))

In [15]:
# analysis.add_filter("prob > 0.01", "prob > 0.01")

In [16]:
for prefix in ["", "postfit_"]:
    analysis.book_histogram_1D(f"{prefix}iso_lep_delta_P", f"{prefix}iso_lep_delta_P", ("", "", 150, -5., 5.,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}nu_delta_P", f"{prefix}nu_delta_P", ("", "", 150, -25., 25.,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet1_delta_P", f"{prefix}R2Jet1_delta_P", ("", "", 150, -15., 15.,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet2_delta_P", f"{prefix}R2Jet2_delta_P", ("", "", 150, -15., 15.,), categories=signal_category)

    analysis.book_histogram_1D(f"{prefix}iso_lep_delta_theta", f"{prefix}iso_lep_delta_theta", ("", "", 150, -0.0005, 0.0005,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}nu_delta_theta", f"{prefix}nu_delta_theta", ("", "", 150, -0.2, 0.2,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet1_delta_theta", f"{prefix}R2Jet1_delta_theta", ("", "", 150, -0.15, 0.15,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet2_delta_theta", f"{prefix}R2Jet2_delta_theta", ("", "", 150, -0.15, 0.15,), categories=signal_category)

    analysis.book_histogram_1D(f"{prefix}iso_lep_delta_phi", f"{prefix}iso_lep_delta_phi", ("", "", 150, -0.0005, 0.0005,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}nu_delta_phi", f"{prefix}nu_delta_phi", ("", "", 150, -0.2, 0.2,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet1_delta_phi", f"{prefix}R2Jet1_delta_phi", ("", "", 150, -0.15, 0.15,), categories=signal_category)
    analysis.book_histogram_1D(f"{prefix}R2Jet2_delta_phi", f"{prefix}R2Jet2_delta_phi", ("", "", 150, -0.15, 0.15,), categories=signal_category)

In [17]:
analysis.book_histogram_1D("prefit_leptonic_W_M", "prefit_leptonic_W_M", ("", "", 250, 0., 250.), categories=signal_category)
analysis.book_histogram_1D("prefit_hadronic_W_M", "prefit_hadronic_W_M", ("", "", 250, 0., 250.), categories=signal_category)
analysis.book_histogram_1D("postfit_leptonic_W_M", "postfit_leptonic_W_M", ("", "", 250, 0., 250.), categories=signal_category)
analysis.book_histogram_1D("postfit_hadronic_W_M", "postfit_hadronic_W_M", ("", "", 250, 0., 250.), categories=signal_category)

In [18]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec)

Info in <[ROOT.RDF] Info /tmp/root/spack-stage/spack-stage-root-6.38.00-2jf5cmbvudyjye7uzxzvsgmmlw2msfso/spack-build-2jf5cmb/include/ROOT/RDF/RInterface.hxx:1363 in auto ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager, void>::Snapshot(std::string_view, std::string_view, const ColumnNames_t &, const RSnapshotOptions &)::(anonymous class)::operator()() const [Proxied = ROOT::Detail::RDF::RLoopManager, DataSource = void]>: 
	In ROOT 6.38, the default compression settings of Snapshot have been changed from 101 (ZLIB with compression level 1, the TTree default) to 505 (ZSTD with compression level 5). This change may result in smaller Snapshot output dataset size by default. In order to suppress this message, set 'ROOT_RDF_SNAPSHOT_INFO=0' in your environment or set 'ROOT.RDF.Snapshot.Info: 0' in your .rootrc file.


In [19]:
analysis.book_reports()

In [20]:
%%time
analysis.run()

CPU times: user 52.7 s, sys: 701 ms, total: 53.4 s
Wall time: 1min 5s


In [21]:
# analysis.print_reports()

In [22]:
analysis.draw_histogram("prob", categories=signal_category)
analysis.draw_histogram("chi2")
analysis.draw_histogram("error")

(<cppyy.gbl.THStack object at 0x12d90620>,
 <cppyy.gbl.TCanvas object at 0x1515cec0>)

In [23]:
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_P", "postfit_iso_lep_delta_P"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_P", "postfit_nu_delta_P"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet1_delta_P", "postfit_R2Jet1_delta_P"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet2_delta_P", "postfit_R2Jet2_delta_P"], category=signal_category[0])

In [24]:
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_theta", "postfit_iso_lep_delta_theta"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_theta", "postfit_nu_delta_theta"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet1_delta_theta", "postfit_R2Jet1_delta_theta"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet2_delta_theta", "postfit_R2Jet2_delta_theta"], category=signal_category[0])

In [25]:
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_phi", "postfit_iso_lep_delta_phi"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_phi", "postfit_nu_delta_phi"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet1_delta_phi", "postfit_R2Jet1_delta_phi"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet2_delta_phi", "postfit_R2Jet2_delta_phi"], category=signal_category[0])

In [26]:
analysis.compare_summed_histograms_unscaled(["prefit_leptonic_W_M", "postfit_leptonic_W_M"], category=signal_category[0])
analysis.compare_summed_histograms_unscaled(["prefit_hadronic_W_M", "postfit_hadronic_W_M"], category=signal_category[0])